# 통합 실행과 운영 한계

허용된 보고서 쓰기와 차단된 파일 쓰기를 함께 실행하고 종료 산출물을 검증합니다.

## 설치

고정된 패키지를 현재 커널에 설치합니다.

In [1]:
# 실습에서 검증한 패키지 버전을 현재 커널에 설치합니다.
%pip install -q deepagents==0.7.0 langchain==1.4.0 langchain-openai==1.6.0 python-dotenv==1.2.3

Note: you may need to restart the kernel to use updated packages.

## 실행 환경 준비

상위 폴더의 .env를 찾아 키 값은 출력하지 않고 버전만 확인합니다.


In [2]:
from importlib.metadata import version
from pathlib import Path
from dotenv import find_dotenv, load_dotenv

# 현재 폴더부터 상위 폴더까지 .env를 찾고 값은 출력하지 않습니다.
env_path = find_dotenv(usecwd=True)
load_dotenv(env_path)
cwd = Path.cwd().resolve()
LAB = next(
    parent / 'labs/part6/ch27-deepagents'
    for parent in [cwd, *cwd.parents]
    if (parent / 'labs/part6/ch27-deepagents/pyproject.toml').is_file()
)
print('deepagents', version('deepagents'))
print('langchain', version('langchain'))
assert version('deepagents') == '0.7.0'

deepagents 0.7.0
langchain 1.4.0


## 단계 1

새 작업 폴더와 /blocked.txt 쓰기 거부 규칙을 준비합니다.

### 코드 읽기와 비교 경계

권한 규칙은 모델에게 부탁하는 문장이 아니라 도구 실행 직전에 판정되는 실행 경계여야 합니다. 이 예제는 허용된 보고서 경로와 차단된 `/blocked.txt` 경로를 각각 요청하고, 도구 결과에서 성공과 `permission denied`를 확인합니다. 모델이 차단 파일을 쓰려고 시도했다는 사실과 파일이 실제로 만들어졌다는 사실은 다릅니다. 거부 결과가 메시지에 남아도 최종 답변이 성공처럼 말할 수 있으므로 파일 존재와 도구 결과를 함께 검사합니다.

정지 조건도 권한과 같은 방식으로 관찰 가능해야 합니다. ‘보고서 작성 완료’는 지정 경로의 파일, 필수 항목, 검증 결과가 모두 있을 때 판정합니다. 최대 반복 수는 무한 실행을 막는 경계일 뿐 업무 완료를 증명하지 않습니다. 이 실습은 사용량을 집계하지 않습니다. 종료 판정을 위해 허용된 파일 본문, 거부 ToolMessage, 차단 파일의 부재를 서로 다른 검사로 남깁니다.


In [3]:
from langchain.chat_models import init_chat_model
from deepagents import create_deep_agent
from deepagents.backends import FilesystemBackend
from deepagents.middleware.filesystem import FilesystemPermission

# 이 노트북만의 새 작업 폴더와 차단 규칙을 준비합니다.
workspace = LAB / 'workspace-08'
workspace.mkdir(parents=True, exist_ok=True)
backend = FilesystemBackend(root_dir=workspace)
deny_blocked = FilesystemPermission(operations=['write'], paths=['/blocked.txt'], mode='deny')
agent = create_deep_agent(model=init_chat_model('openai:gpt-5.4-mini'), backend=backend, permissions=[deny_blocked])

## 단계 2

허용 보고서와 차단 파일에 차례로 쓰도록 요청합니다.

허용 경로와 차단 경로를 한 요청에 넣어 성공과 거부를 함께 관찰합니다. 단순히 최종 답변을 읽지 말고 도구 메시지를 확인해야 합니다.


In [4]:
# 허용 경로에는 보고서를 쓰고 차단 경로에는 쓰기를 시도합니다.
prompt = '''write_file로 /run/report.md에 "status: complete"를 쓰세요.
그 다음 /blocked.txt에도 같은 내용을 쓰려고 시도하고, 두 결과를 설명한 뒤 종료하세요.'''
result = agent.invoke({'messages':[{'role':'user','content':prompt}]})
tool_messages = [m for m in result['messages'] if type(m).__name__ == 'ToolMessage']
for message in tool_messages: print(message.name, str(message.content)[:300])

write_file Updated file /run/report.md
write_file Error: permission denied for write on /blocked.txt


## 단계 3

완료 보고서와 permission denied 도구 결과를 함께 확인합니다.

종료 조건은 허용된 보고서의 내용과 차단 도구의 `permission denied` 결과를 모두 확인합니다. 모델의 완료 문장만으로는 충분하지 않습니다.


In [5]:
# 종료 조건과 권한 거부를 실제 파일과 도구 메시지로 확인합니다.
report = backend.read('/run/report.md')
denied = [m for m in tool_messages if 'permission' in str(m.content).lower() or 'denied' in str(m.content).lower()]
print('보고서:', report.file_data['content'])
print('최종 답변:', result['messages'][-1].text)
assert report.file_data['content'].strip() == 'status: complete'
assert denied
assert result['messages'][-1].text

보고서: status: complete
최종 답변: `/run/report.md`에는 `status: complete`를 정상적으로 썼습니다.

`/blocked.txt`에도 같은 내용을 쓰려고 시도했지만, 쓰기 권한이 없어 `permission denied` 오류가 발생했습니다.
